In [1]:
import os
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import noisereduce as nr
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import joblib

/home/habib/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
training_path  = "/home/habib/mindcloud/project/balanced_train.csv"
balanced_train = pd.read_csv(training_path)
balanced_train_copy = balanced_train.copy()
print(balanced_train_copy["y"].value_counts()['discomfort'])
print(balanced_train_copy["y"].value_counts()['tired'])
print(balanced_train_copy["y"].value_counts()['hungry'])

308
306
306


In [4]:
import warnings

def extract_features_safe(file_path, n_mfcc=13, top_db=20):
    """
    Robust feature extractor returning five features:
      [mfcc_summary, duration_s, rms_mean, f0_median_hz, zcr_mean]

    Safely tries soundfile first, falls back to librosa. Prints clear error messages
    for file-not-found, empty files, and open/read failures.
    """
    # Basic existence + size checks
    if not os.path.exists(file_path):
        print(f"Error processing {file_path}: file not found")
        return None

    try:
        size = os.path.getsize(file_path)
    except Exception as e:
        print(f"Error processing {file_path}: cannot stat file ({e})")
        return None

    if size == 0:
        print(f"Error processing {file_path}: file is empty (0 bytes)")
        return None

    # Try to read with soundfile (more explicit errors); fallback to librosa.load (audioread)
    y = None
    sr = None
    try:
        # soundfile returns shape (nsamples, nchannels) or (nsamples,)
        y, sr = sf.read(file_path, dtype='float32')
        if y is None:
            raise RuntimeError("soundfile returned None")
        # if multi-channel, convert to mono by averaging channels
        if y.ndim > 1:
            y = np.mean(y, axis=1)
    except Exception as e_sf:
        # fallback: librosa.load (audioread backend). More forgiving but less explicit.
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                y, sr = librosa.load(file_path, sr=None, mono=True)
        except Exception as e_lb:
            print(f"Error processing {file_path}: could not open audio file; soundfile error: {e_sf}; librosa error: {e_lb}")
            return None

    # If still nothing
    if y is None or sr is None:
        print(f"Error processing {file_path}: failed to load (unknown reason)")
        return None

    # Trim silence (top_db)
    try:
        y, _ = librosa.effects.trim(y, top_db=top_db)
    except Exception:
        # If trim fails, continue with original signal
        pass

    # Clean NaNs/Infs (can appear after augmentations)
    if np.isnan(y).any() or np.isinf(y).any():
        y = np.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0)

    if len(y) == 0 or np.all(y == 0):
        print(f"Error processing {file_path}: audio empty after trimming / cleaning")
        return None

    # 1) Duration (seconds)
    duration = float(len(y)) / float(sr)

    # 2) MFCC summary: compute n_mfcc and reduce to a single scalar
    try:
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
        mfcc_mean_per_coeff = np.mean(mfccs, axis=1)
        mfcc_summary = float(np.mean(mfcc_mean_per_coeff))
    except Exception:
        # if MFCC fails, set a safe default and continue
        mfcc_summary = 0.0

    # 3) RMS intensity: mean across frames
    try:
        rms = librosa.feature.rms(y=y)
        rms_mean = float(np.mean(rms)) if rms.size > 0 else 0.0
    except Exception:
        rms_mean = 0.0

    # 4) Fundamental frequency (F0): try librosa.yin (median of finite estimates)
    f0_median = 0.0
    try:
        fmin = 50.0
        fmax = min(2000.0, sr / 2.0 - 1.0)
        # Ensure a reasonable fmax
        if fmax > fmin:
            f0_candidates = librosa.yin(y, fmin=fmin, fmax=fmax, sr=sr)
            f0_finite = f0_candidates[np.isfinite(f0_candidates)]
            if f0_finite.size > 0:
                f0_median = float(np.median(f0_finite))
    except Exception:
        f0_median = 0.0

    # 5) Zero-Crossing Rate (ZCR): mean across frames
    try:
        zcr = librosa.feature.zero_crossing_rate(y)
        zcr_mean = float(np.mean(zcr)) if zcr.size > 0 else 0.0
    except Exception:
        zcr_mean = 0.0

    features = np.array([mfcc_summary, duration, rms_mean, f0_median, zcr_mean], dtype=float)
    return features


def list_unreadable_wavs(root_dir):
    """
    Quick scanner to list WAV files that fail the basic open check using soundfile.info.
    This is fast and avoids full feature extraction; use it to find corrupted/empty files.
    """
    unreadable = []
    for dirpath, _, files in os.walk(root_dir):
        for fname in files:
            if not fname.lower().endswith(('.wav', '.flac', '.mp3', '.ogg')):
                continue
            path = os.path.join(dirpath, fname)
            try:
                # quick info check
                sf.info(path)
            except Exception as e:
                unreadable.append((path, str(e)))
    for p, err in unreadable:
        print(f"Unreadable: {p}  -> {err}")
    return unreadable

In [5]:
testing_path = "/home/habib/mindcloud/project/test.csv"
test = pd.read_csv(testing_path)
test_copy = test.copy()

In [6]:
# Replace your current model line in Cell 6 with this tuned version:
model = RandomForestClassifier(
    n_estimators=150,           # More trees provide more stable voting bounds
    max_depth=12,               # Strictly caps depth to stop trees from memorizing individual files
    min_samples_split=4,        # Prevents creating splits for ultra-specific audio quirks
    max_features="log2",        # Limits feature scanning per split, forcing trees to consider minor MFCC values
    class_weight="balanced",     # Maintains minority penalty adjustment
    random_state=42
)

In [7]:
# -------------------------------------------------------------------------
# A. RUN SAFE FEATURE EXTRACTION
# -------------------------------------------------------------------------
X_train_list, y_train_list = [], []
print("Extracting features from balanced training dataset...")
for idx, row in balanced_train_copy.iterrows():
    feats = extract_features_safe(row['x'])
    if feats is not None:
        X_train_list.append(feats)
        y_train_list.append(row['y'])

X_test_list, y_test_list = [], []
print("Extracting features from unseen test set...")
for idx, row in test_copy.iterrows():
    feats = extract_features_safe(row['x'])
    if feats is not None:
        X_test_list.append(feats)
        y_test_list.append(row['y'])

X_train, y_train = np.array(X_train_list), np.array(y_train_list)
X_test, y_test = np.array(X_test_list), np.array(y_test_list)

# -------------------------------------------------------------------------
# B. TRAIN AND EVALUATE MODEL
# -------------------------------------------------------------------------
print("\nTraining the Random Forest model...")
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(f"\n True Test Set Accuracy: {model.score(X_test, y_test) * 100:.2f}%")
print("\n--- Corrected Classification Report ---")
print(classification_report(y_test, y_pred))
print("--- Confusion Matrix ---")
print(confusion_matrix(y_test, y_pred))

Extracting features from balanced training dataset...


Error processing /home/habib/mindcloud/project/dataset/augmented/aug_tired_418_15.wav: file not found
Error processing /home/habib/mindcloud/project/dataset/augmented/aug_tired_424_9.wav: file not found
Error processing /home/habib/mindcloud/project/dataset/augmented/aug_tired_424_2.wav: file not found
Error processing /home/habib/mindcloud/project/dataset/augmented/aug_discomfort_16_3.wav: file not found
Error processing /home/habib/mindcloud/project/dataset/augmented/aug_discomfort_25_9.wav: file not found
Error processing /home/habib/mindcloud/project/dataset/augmented/aug_tired_414_5.wav: file not found
Error processing /home/habib/mindcloud/project/dataset/augmented/aug_discomfort_0_8.wav: file not found
Error processing /home/habib/mindcloud/project/dataset/augmented/aug_discomfort_12_11.wav: file not found
Error processing /home/habib/mindcloud/project/dataset/augmented/aug_discomfort_9_5.wav: file not found
Error processing /home/habib/mindcloud/project/dataset/augmented/aug_di

KeyboardInterrupt: 

In [8]:
print(test_copy["y"].value_counts()['discomfort'])
print(test_copy["y"].value_counts()['tired'])
print(test_copy["y"].value_counts()['hungry'])

5
6
76


In [7]:
"""# -------------------------------------------------------------------------
# 4. SAVE THE MODEL
# -------------------------------------------------------------------------
model_filename = "audio_random_forest_model_acc_86%.joblib"
joblib.dump(model, model_filename)
print(f"\nModel successfully saved to {model_filename}")"""

'# -------------------------------------------------------------------------\n# 4. SAVE THE MODEL\n# -------------------------------------------------------------------------\nmodel_filename = "audio_random_forest_model_acc_86%.joblib"\njoblib.dump(model, model_filename)\nprint(f"\nModel successfully saved to {model_filename}")'